# Isolation Forest
        **unsupervised › anomaly_detection** · notebook `isolation_forest`

        **Kapan dipakai:** Pilihan pertama untuk data tabular: cepat, tanpa asumsi distribusi, skala besar.

        **Jebakan umum:** Kurang baik bila anomali berada di subruang tertentu (fitur banyak yang tidak relevan).

**Data demo:** Breast cancer: 357 jinak (normal) + 22 ganas (anomali). Label hanya untuk evaluasi; model tidak melihatnya (kecuali varian semi-supervised yang hanya memakai data normal).

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/breast_cancer_anomaly.csv"; LABEL_COL = "is_anomaly"; TEST_SIZE = 0.3
MODEL_KEY = "isolation_forest"
OUT = out_dir("unsupervised", "anomaly_detection", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Data dan pemisahan
Train = tanpa label (dianggap tidak berlabel; mungkin tercemar anomali). Test = dipakai menilai seberapa baik skor memisahkan anomali.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from superdb.unsupervised import anomaly_eval
df = pd.read_csv(PATH); y = df[LABEL_COL].to_numpy(); X = df.drop(columns=[LABEL_COL]).select_dtypes("number").to_numpy(float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
sc = StandardScaler().fit(Xtr); Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
print(f"train {Xtr.shape} (anomali tersembunyi: {int(ytr.sum())}) | test {Xte.shape} (anomali: {int(yte.sum())}, {yte.mean():.1%})")

## 3. Skor anomali
Semua metode berbentuk `score_fn(Xfit, Xeval)` yang mengembalikan skor (makin besar = makin aneh).

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

def score_fn(Xfit, Xeval):
    m = IsolationForest(n_estimators=300, random_state=42, n_jobs=-1).fit(Xfit)
    return -m.score_samples(Xeval)

## 4. Evaluasi dua skenario
- **A. Tidak terawasi**: model dilatih pada SEMUA data train (termasuk anomali yang tersembunyi). Realistis bila tak ada label.
- **B. Semi-terawasi (novelty)**: model dilatih HANYA pada data normal. Lebih mudah; hanya mungkin bila ada contoh normal terjamin.

**PR-AUC** adalah metrik utama untuk anomali langka; bandingkan dengan garis dasar (proporsi anomali).

In [ ]:
import time; t0 = time.time()
sA = np.asarray(score_fn(Xtr_s, Xte_s), float); tA = time.time() - t0
sB = np.asarray(score_fn(Xtr_s[ytr == 0], Xte_s), float)
rA, rB = anomaly_eval(yte, sA), anomaly_eval(yte, sB)
display(pd.DataFrame({"A tidak terawasi": rA, "B semi-terawasi": rB}).round(4)); print(f"waktu skenario A: {tA:.1f} dtk")

## 5. Memilih ambang
Tanpa label, ambang biasanya berupa 'tandai persentase teratas' sesuai perkiraan proporsi anomali atau kapasitas penanganan alarm.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
k = max(1, int(round(ytr.mean() * len(yte))))            # perkiraan proporsi anomali dari data train (dalam praktik: asumsi domain)
flag = np.zeros(len(yte), int); flag[np.argsort(-sA)[:k]] = 1
print(f"menandai {k} teratas: precision={precision_score(yte, flag):.3f} recall={recall_score(yte, flag):.3f} F1={f1_score(yte, flag):.3f}")
from sklearn.metrics import PrecisionRecallDisplay
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for lab, name, col in ((0, "normal", "tab:blue"), (1, "anomali", "tab:red")): ax[0].hist(sA[yte == lab], bins=25, alpha=.6, label=name, color=col, density=True)
ax[0].legend(); ax[0].set_title("distribusi skor (A)")
PrecisionRecallDisplay.from_predictions(yte, sA, ax=ax[1], name="A"); PrecisionRecallDisplay.from_predictions(yte, sB, ax=ax[1], name="B"); ax[1].axhline(yte.mean(), color="gray", ls=":")
plt.tight_layout(); plt.savefig(OUT / "anomaly.png", dpi=130); plt.show()
pd.DataFrame({"skor_A": sA, "skor_B": sB, "label": yte}).to_csv(OUT / "scores_test.csv", index=False)
save_json({"A_unsupervised": rA, "B_semi_supervised": rB, "top_k_flag": {"k": k, "precision": float(precision_score(yte, flag)), "recall": float(recall_score(yte, flag))}}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Data lomba biasanya tanpa label: pilih metode lewat kecocokan jenis data (musiman? berdimensi tinggi? multimodal?), bukan satu skor.
- Gabungkan skor beberapa metode (rata-rata rank) untuk keandalan.
- Analisis kasus yang ditandai teratas secara manual: itu bahan analisis laporan.